# Modelling — Krakow PM2.5 Spatial Regression

**Purpose:** take the joined training dataset (aqicn monthly means + Sentinel-2 / OSM / ERA5 features)
from Session 3 and produce a defensible baseline + one honest model, with a model card.

**Input:** `data/processed/training_data.parquet` (aqicn_monthly joined with spatial features)

**Outputs:**
- `data/processed/aqicn-monthly-{train,val,test}.parquet` — station-group split
- `models/baseline.joblib` — saved sklearn Pipeline
- `docs/model-cards/krakow-pm25-spatial-rf-v1.md` — the model certificate
- `docs/modelling-log.md` — every decision

**Split:** leave-one-station-out (spatial group). Tests generalization to new locations — the only split that matches the deployment scenario (32,700 grid cells with no PM2.5 monitors).

This notebook is the **exploratory** layer. Stable logic is promoted to `src/split_data.py` and `src/baseline_model.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import joblib
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Paths — relative to notebooks/ directory
CLEAN_PATH = Path("../data/processed/training_data.parquet")
SPLIT_DIR  = Path("../data/processed/")
MODEL_PATH = Path("../models/baseline.joblib")

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Target and features (from Session 3 pipeline design)
TARGET = "pm25_mean"
FEATURE_COLS = [
    "ndvi",
    "ndbi",
    "road_density_500m",
    "building_density_500m",
    "pct_green_500m",
    "mean_temp_monthly",
    "blh_monthly",
    "month",   # derived from year_month — seasonal control
]

# Station split assignments — fixed for reproducibility
TEST_STATIONS = frozenset({"zloty_rog", "nowa_huta"})
VAL_STATIONS  = frozenset({"kurdwanow"})

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Imports and config OK")

In [ ]:
# Load joined dataset (aqicn_monthly + spatial features)
df = pd.read_parquet(CLEAN_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Stations ({df['station_id'].nunique()}): {sorted(df['station_id'].unique())}")
print(f"Year-months: {df['year_month'].min()} → {df['year_month'].max()}")
print(f"Target (pm25_mean) — min: {df['pm25_mean'].min():.1f}  max: {df['pm25_mean'].max():.1f}  mean: {df['pm25_mean'].mean():.1f} µg/m³")
df.head()

> **Discipline:** start every modelling notebook by re-stating the contract from S3.
> Expected: 10 stations, ~600 station-months (2019-01 to 2024-12 minus COVID months and low-completeness station-months).
> Expected columns: 13 from aqicn_monthly.csv + 7 from feature extraction = 20 total.
> If anything looks off (row count, missing feature columns, NaN-heavy columns), stop and check `docs/data-cleaning-log.md` and `notebooks/03-feature-extraction.ipynb`.

In [ ]:
# Quick S3 contract audit
assert df['station_id'].nunique() == 10, f"Expected 10 stations, got {df['station_id'].nunique()}"
assert len(df) >= 550, f"Expected ~600 station-months, got {len(df)}"
assert (df['pm25_mean'] >= 0).all(), "Negative pm25_mean — cleaning contract broken"

# Check feature columns are present
missing_features = [c for c in FEATURE_COLS if c not in df.columns and c != 'month']
if missing_features:
    print(f"WARNING: Missing feature columns (run feature extraction first): {missing_features}")
else:
    print("All feature columns present")

# Derive month feature
df['month'] = df['year_month'].str[5:7].astype(int)

# COVID check: no 2020-03, 2020-04, 2020-05
covid_months = {'2020-03', '2020-04', '2020-05'}
overlap = set(df['year_month'].unique()) & covid_months
assert not overlap, f"COVID months in cleaned data: {overlap}"
print(f"COVID exclusion confirmed. Shape: {df.shape}")

# NaN check per feature
nan_pct = df[FEATURE_COLS].isna().mean().sort_values(ascending=False)
print("\nNaN % per feature column:")
print(nan_pct[nan_pct > 0].to_string() or "  (no NaNs)")

## Task 1: Split

**Decision: leave-one-station-out (LOSO) group split.**

**Why not random / temporal:**
- PM2.5 autocorrelation at lag-1 month within a station is ~0.65. A random split puts sibling months from the same station in both train and test — leaking the station-level mean PM2.5.
- A temporal cutoff puts all 10 stations in test — the model is then evaluated on stations it was trained on. That tests temporal generalisation, not spatial.
- **Our model predicts at 32,700 grid cells with no monitors.** LOSO is the only split that tests the right question: can this model predict at a location it has never seen?

**Station assignments (fixed, not randomised):**
- `test`: `{zloty_rog, nowa_huta}` — SACRED until model is locked
- `val`: `{kurdwanow}` — used for all hyperparameter choices
- `train`: remaining 7 stations

**Logged in:** `docs/modelling-log.md` decision #1.

In [ ]:
# Station-group split
train_stations = set(df['station_id'].unique()) - TEST_STATIONS - VAL_STATIONS

train = df[df['station_id'].isin(train_stations)].copy().reset_index(drop=True)
val   = df[df['station_id'].isin(VAL_STATIONS)].copy().reset_index(drop=True)
test  = df[df['station_id'].isin(TEST_STATIONS)].copy().reset_index(drop=True)

# Leakage assertions — the contract
train_g = set(train['station_id'])
val_g   = set(val['station_id'])
test_g  = set(test['station_id'])
assert not (train_g & val_g),  f"Station leak train↔val: {train_g & val_g}"
assert not (train_g & test_g), f"Station leak train↔test: {train_g & test_g}"
assert not (val_g & test_g),   f"Station leak val↔test: {val_g & test_g}"

print(f"train: {len(train_stations)} stations · {len(train):,} rows")
print(f"  stations: {sorted(train_stations)}")
print(f"val:   {len(VAL_STATIONS)} station  · {len(val):,} rows")
print(f"  stations: {sorted(VAL_STATIONS)}")
print(f"test:  {len(TEST_STATIONS)} stations · {len(test):,} rows  ← SACRED")
print(f"  stations: {sorted(TEST_STATIONS)}")
print("\nAll leakage assertions passed.")

In [ ]:
# Write the three split parquets
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
SLUG = "aqicn-monthly"

for name, part in [("train", train), ("val", val), ("test", test)]:
    out = SPLIT_DIR / f"{SLUG}-{name}.parquet"
    part.to_parquet(out, index=False)
    print(f"Wrote {out}  ({len(part):,} rows · {out.stat().st_size / 1024:.0f} KB)")

print("\nTest parquet is now sacred. Do NOT open it until the model is locked.")

## Task 2: Baselines — the floor

Two baselines before any 'real' model:

1. **Dumb mean:** predict the training-set mean pm25_mean (~35 µg/m³) for every row. The unconditional floor — beats nothing but anchors the scale.
2. **Seasonal mean:** predict the month-specific mean from training data. Captures the Krakow heating-season signal (winter ~58 µg/m³ vs summer ~12 µg/m³) without any spatial features. **If our model can't beat this, it isn't learning urban form — it's re-discovering that January is polluted.**

Persistence (y_{t-1}) is not used: it requires knowing the previous month's PM2.5 at the same station — not available for unmeasured grid cells at deployment time.

In [ ]:
# Define feature arrays (NaN-safe — imputer handles remaining NaNs in Pipeline)
X_train = train[FEATURE_COLS]
y_train = train[TARGET]
X_val   = val[FEATURE_COLS]
y_val   = val[TARGET]
X_test  = test[FEATURE_COLS]   # defined here but NOT used until cell c19
y_test  = test[TARGET]         # same — sacred

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}    |  y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}   |  y_test:  {y_test.shape}  ← defined but sacred")
print(f"\ny_train stats: mean={y_train.mean():.1f}  std={y_train.std():.1f}  min={y_train.min():.1f}  max={y_train.max():.1f} µg/m³")

In [ ]:
# Baseline 1: dumb mean
dumb = DummyRegressor(strategy="mean")
dumb.fit(X_train, y_train)

dumb_train_mae = mean_absolute_error(y_train, dumb.predict(X_train))
dumb_val_mae   = mean_absolute_error(y_val,   dumb.predict(X_val))
dumb_train_r2  = r2_score(y_train, dumb.predict(X_train))
dumb_val_r2    = r2_score(y_val,   dumb.predict(X_val))

print(f"Dumb-mean baseline:")
print(f"  train MAE = {dumb_train_mae:.2f} µg/m³   train R² = {dumb_train_r2:.3f}")
print(f"  val   MAE = {dumb_val_mae:.2f} µg/m³   val   R² = {dumb_val_r2:.3f}")
print(f"  (Predicts {dumb.constant_[0]:.1f} µg/m³ for every row)")

In [ ]:
# Baseline 2: seasonal mean (month-specific average from training data)
month_means = train.groupby("month")[TARGET].mean()
print("Training month means (µg/m³):")
print(month_means.round(1).to_string())

def predict_seasonal(month_means, df):
    overall = month_means.mean()
    return df["month"].map(month_means).fillna(overall)

seas_train_preds = predict_seasonal(month_means, train)
seas_val_preds   = predict_seasonal(month_means, val)

seas_train_mae = mean_absolute_error(y_train, seas_train_preds)
seas_val_mae   = mean_absolute_error(y_val,   seas_val_preds)
seas_train_r2  = r2_score(y_train, seas_train_preds)
seas_val_r2    = r2_score(y_val,   seas_val_preds)

print(f"\nSeasonal-mean baseline:")
print(f"  train MAE = {seas_train_mae:.2f} µg/m³   train R² = {seas_train_r2:.3f}")
print(f"  val   MAE = {seas_val_mae:.2f} µg/m³   val   R² = {seas_val_r2:.3f}")
print(f"\nSeasonal beats dumb by {dumb_val_mae - seas_val_mae:.1f} µg/m³ on val.")
print("Our model must beat the seasonal baseline on val to prove it learns urban form.")

## Task 3: One defensible model

**Three questions before picking a technique:**

1. **Problem shape:** regression — predict monthly mean PM2.5 from 8 mixed numeric features.
2. **Data size & dim:** ~420 training rows × 8 features. Too small for deep learning. Large enough for a Random Forest to generalise if min_samples_leaf prevents over-growth.
3. **Will I be asked to explain this?** Yes — the Session 7 dashboard shows the planner which features drove the prediction for their parcel. SHAP feature importances for RF are well-supported and interpretable.

**Chosen technique:** `RandomForestRegressor(n_estimators=300, min_samples_leaf=3)`

**Why not linear regression:** PM2.5 ~ BLH (boundary layer height) is nonlinear — very high BLH suppresses PM2.5 non-proportionally during summer. Ridge R² on val expected ~0.30, below the ≥0.40 brief target.

**Why not Gradient Boosting:** Would likely outperform RF. Deferred to Session 5 — we establish the defensible baseline here, not the optimal model.

**Logged in:** `docs/modelling-log.md` decisions #5 (class), #6 (hyperparameters).

In [ ]:
# Build and fit the Pipeline
pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),   # safety net for NaN features
    ("scale",  StandardScaler()),                   # RF is scale-invariant but SHAP values are cleaner
    ("model",  RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

pipeline.fit(X_train, y_train)
print("Pipeline fitted.")
print(f"  Steps: {[s[0] for s in pipeline.steps]}")
print(f"  n_features: {pipeline.n_features_in_}")

In [ ]:
# Score model on train + val (NOT test yet)
model_train_mae = mean_absolute_error(y_train, pipeline.predict(X_train))
model_val_mae   = mean_absolute_error(y_val,   pipeline.predict(X_val))
model_train_r2  = r2_score(y_train, pipeline.predict(X_train))
model_val_r2    = r2_score(y_val,   pipeline.predict(X_val))

print(f"RF model:")
print(f"  train MAE = {model_train_mae:.2f} µg/m³   train R² = {model_train_r2:.3f}")
print(f"  val   MAE = {model_val_mae:.2f} µg/m³   val   R² = {model_val_r2:.3f}")

# Gap check
gap = model_val_mae - model_train_mae
if gap > 10:
    print(f"\nWARNING: Large train↔val gap ({gap:.1f} µg/m³). Possible overfit — consider increasing min_samples_leaf.")
elif model_val_mae < model_train_mae * 0.5:
    print(f"\nWARNING: val MAE ({model_val_mae:.2f}) < half of train MAE ({model_train_mae:.2f}). Possible leakage.")
else:
    print(f"  Train↔val gap: {gap:.1f} µg/m³ — within expected range.")

# Success criterion check
if model_val_r2 >= 0.40:
    print(f"\nBrief success criterion met: val R² = {model_val_r2:.3f} ≥ 0.40")
else:
    print(f"\nBrief success criterion NOT YET met: val R² = {model_val_r2:.3f} < 0.40")
    print("  Options: add more features, tune hyperparameters in Session 5, or accept and document.")

In [ ]:
# Honest metrics table — train + val only (test sacred)
report = pd.DataFrame([
    {"split": "train", "model": "dumb-mean",  "MAE": round(dumb_train_mae, 2),  "R2": round(dumb_train_r2, 3)},
    {"split": "train", "model": "seasonal",   "MAE": round(seas_train_mae, 2),  "R2": round(seas_train_r2, 3)},
    {"split": "train", "model": "RF (ours)",  "MAE": round(model_train_mae, 2), "R2": round(model_train_r2, 3)},
    {"split": "val",   "model": "dumb-mean",  "MAE": round(dumb_val_mae, 2),    "R2": round(dumb_val_r2, 3)},
    {"split": "val",   "model": "seasonal",   "MAE": round(seas_val_mae, 2),    "R2": round(seas_val_r2, 3)},
    {"split": "val",   "model": "RF (ours)",  "MAE": round(model_val_mae, 2),   "R2": round(model_val_r2, 3)},
])
print(report.to_markdown(index=False))
print("\nNote: test split NOT computed here. Test is touched once — see cell c19.")

> **Discipline:** the test row is absent. We do NOT compute test metrics until the model is locked — hyperparameters chosen, code committed. Touching test before that point contaminates the evaluation. The table above is what goes in the model card section 7.1 (minus the test rows, which are added later).

## Task 4: Assess — the honest part

### 4a: Leave-one-station-out cross-validation

Use all 8 train+val stations for LOSO-CV to get the spread (mean ± std) across urban typologies. This is the number that goes in the model card as the 'val LOSO' row.

In [ ]:
# LOSO-CV on train+val (never touch test)
train_val = pd.concat([train, val], ignore_index=True)
loso_stations = train_val['station_id'].unique()

fold_rows = []
for held_out in loso_stations:
    fold_train = train_val[train_val['station_id'] != held_out]
    fold_val   = train_val[train_val['station_id'] == held_out]

    p = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
        ("model",  RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                          random_state=RANDOM_SEED, n_jobs=-1)),
    ])
    p.fit(fold_train[FEATURE_COLS], fold_train[TARGET])
    preds = p.predict(fold_val[FEATURE_COLS])

    fold_rows.append({
        "held_out": held_out,
        "n_val":    len(fold_val),
        "MAE":      round(mean_absolute_error(fold_val[TARGET], preds), 2),
        "R2":       round(r2_score(fold_val[TARGET], preds), 3),
    })

cv_results = pd.DataFrame(fold_rows).sort_values("MAE", ascending=False)
print(cv_results.to_markdown(index=False))
print(f"\nLOSO-CV MAE: {cv_results['MAE'].mean():.2f} ± {cv_results['MAE'].std():.2f} µg/m³")
print(f"LOSO-CV R²:  {cv_results['R2'].mean():.3f} ± {cv_results['R2'].std():.3f}")
print("\nWorst stations (first candidates for Session 6 failure gallery):")
print(cv_results.head(3)[['held_out', 'MAE', 'R2']].to_string(index=False))

In [ ]:
# Uncertainty intervals from tree quantiles
forest = pipeline.named_steps["model"]
Xt_val = pipeline[:-1].transform(X_val)  # preprocessed val features

all_tree_preds = np.stack([t.predict(Xt_val) for t in forest.estimators_])
y_pred_lo = np.percentile(all_tree_preds, 5,  axis=0)
y_pred_hi = np.percentile(all_tree_preds, 95, axis=0)
y_pred    = all_tree_preds.mean(axis=0)

coverage = np.mean((y_val.values >= y_pred_lo) & (y_val.values <= y_pred_hi))
interval_width = (y_pred_hi - y_pred_lo).mean()

print(f"90% tree-quantile interval on val (Kurdwanów):")
print(f"  Coverage: {coverage:.1%}  (target ≥ 85%)")
print(f"  Mean interval width: {interval_width:.1f} µg/m³")
if coverage < 0.85:
    print("  WARNING: Under-coverage — intervals are overconfident. Consider conformal prediction in Session 5.")

# Plot: actual vs predicted with intervals
fig, ax = plt.subplots(figsize=(10, 4))
months = val['year_month'].values
ax.plot(months, y_val.values, 'k-o', ms=4, label='Observed')
ax.plot(months, y_pred, 'b--', label='RF predicted (mean)')
ax.fill_between(months, y_pred_lo, y_pred_hi, alpha=0.25, color='blue', label='90% interval')
ax.set_xlabel('Month'); ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('Val station (Kurdwanów) — observed vs predicted with uncertainty')
ax.legend(); plt.xticks(rotation=45, ha='right', fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
# Per-segment performance on val (and LOSO-CV results)
val_pred = pipeline.predict(X_val)
val_with_pred = val.assign(pred=val_pred, abs_err=np.abs(val_pred - y_val))

# Per-season performance on val
val_with_pred['season'] = val_with_pred['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn',
})
per_season = val_with_pred.groupby('season')['abs_err'].agg(['mean','count']).rename(columns={'mean':'MAE','count':'n'})
print("Per-season MAE on val (Kurdwanów):")
print(per_season.round(2).to_string())

# Per-station MAE from LOSO-CV (the most important per-segment view)
print("\nPer-station MAE from LOSO-CV (all 8 train+val stations):")
print(cv_results[['held_out','MAE','R2','n_val']].to_string(index=False))

# Feature importance
importances = pd.Series(
    pipeline.named_steps['model'].feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)
print("\nFeature importances (RF MDI):")
print(importances.round(3).to_string())

> **This is where Session 6 starts.** Note the worst station from LOSO-CV — that's the failure gallery's first entry. Note the worst season — that's the second. If `mean_temp_monthly` or `blh_monthly` dominates feature importance, the model is mostly learning the heating season, not urban form. Check that `month` feature importance is reasonable but not overwhelming.

## Task 5: Lock the model — touch test once

In [ ]:
# Save the pipeline and run round-trip check
joblib.dump(pipeline, MODEL_PATH)

loaded = joblib.load(MODEL_PATH)
assert np.allclose(loaded.predict(X_val), pipeline.predict(X_val)), \
    "saved pipeline doesn't reproduce in-memory predictions"

print(f"Saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.1f} KB)")
print("Round-trip predictions match in-memory: OK")
print("\nModel is now LOCKED. Only run the next cell (c19) once.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL TEST SCORE — run this cell ONCE, at the very end, after model is locked
# Model is committed. Hyperparameters are frozen. Do NOT re-tune after seeing these.
# ─────────────────────────────────────────────────────────────────────────────

test_pred = pipeline.predict(X_test)
test_mae  = mean_absolute_error(y_test, test_pred)
test_r2   = r2_score(y_test, test_pred)

dumb_test_mae  = mean_absolute_error(y_test, dumb.predict(X_test))
seas_test_preds = predict_seasonal(month_means, test)
seas_test_mae  = mean_absolute_error(y_test, seas_test_preds)

# Uncertainty on test
Xt_test = pipeline[:-1].transform(X_test)
all_tree_test = np.stack([t.predict(Xt_test) for t in forest.estimators_])
test_lo = np.percentile(all_tree_test, 5, axis=0)
test_hi = np.percentile(all_tree_test, 95, axis=0)
test_coverage = np.mean((y_test.values >= test_lo) & (y_test.values <= test_hi))

print("══════════════════════════════════════════════")
print("FINAL TEST RESULTS (Złoty Róg + Nowa Huta)")
print("══════════════════════════════════════════════")
print(f"  dumb-mean  MAE: {dumb_test_mae:.2f} µg/m³")
print(f"  seasonal   MAE: {seas_test_mae:.2f} µg/m³")
print(f"  RF (ours)  MAE: {test_mae:.2f} µg/m³    R²: {test_r2:.3f}")
print(f"  90% interval coverage: {test_coverage:.1%}")
print()
if test_r2 >= 0.40:
    print(f"Brief success criterion MET: test R² = {test_r2:.3f} ≥ 0.40")
else:
    print(f"Brief success criterion NOT MET: test R² = {test_r2:.3f} < 0.40")
    print("  → Document in model card. Investigate in Session 5/6.")
print()
print("NOTE: Złoty Róg reads 20-30% low (sheltered placement bias).")
print("      Test MAE is partly driven by the sensor bias, not model error.")
print("      See model card section 3 and data-cleaning-log.md Transform 7.")
print("══════════════════════════════════════════════")

## What gets promoted to `src/`?

Two modules emerge from this notebook:

**`src/split_data.py`** — `make_station_splits`, `make_loso_folds`, `write_splits`, leakage assertions. Importable from this notebook after promotion:
```python
from src.split_data import make_station_splits, write_splits
result = make_station_splits(df)
write_splits(result)
```

**`src/baseline_model.py`** — `build_pipeline`, `train_baseline_model`, `run_loso_cv`, `compute_metrics_table`, `predict_with_uncertainty`, `save_pipeline`. Importable after promotion:
```python
from src.baseline_model import train_baseline_model, predict_with_uncertainty
pipeline = train_baseline_model(X_train, y_train)
point, lo, hi = predict_with_uncertainty(pipeline, X_val)
```

**Pre-commit ritual:**
1. Restart kernel → Run All — no errors
2. `python src/split_data.py` — parquets regenerate, all assertions pass
3. `python src/baseline_model.py` — `models/baseline.joblib` regenerates, round-trip check passes
4. Metrics table in this notebook matches what the script prints
5. Model card section 7 filled with the numbers printed in cell c19